# Пилот корпуса Википедии

Собираем статьи Википедии по 8 пилотным темам (`data/tasks/pilot_topics.yaml`) и сохраняем выдачу
живого поиска Википедии: по ней потом подбираются веса нашего BM25 и проверяется, находятся ли разделы
с понятиями из тезисов.

Запросы по теме: сама тема и ключевые слова тезисов (`terms`) трёх составителей (пилот 3) — короткие
запросы: длинные фразы поиск Википедии понимает плохо (первый запуск, по заголовкам тезисов, это показал).
Для каждой темы берём статьи из выдачи русской Википедии, статьи из ссылок во введениях трёх лучших статей
по теме, английские версии найденных статей по межъязыковым ссылкам и выдачу английской Википедии по
английским названиям трёх лучших статей.
Код — `src/presentation_rl/tools/wiki.py` и `tools/corpus.py`.

Нужен интернет, видеокарта не нужна. Около 1000–1500 запросов к API, примерно 10–15 минут
(Википедия просит не спешить: пауза 0,5 с между запросами). Сборку можно прервать и запустить снова —
скачанное повторно не запрашивается.

Википедия требует указать контакт в заголовке запросов: перед запуском задайте переменную окружения
`WIKI_CONTACT` (почта или ссылка на страницу проекта) или впишите контакт в `CONTACT` ниже.

Результат — папка `data/sources/wiki_pilot/` (в git не попадает): `articles.jsonl` (общий для запусков:
уже скачанные статьи повторно не качаются) и `topics_v2.jsonl` (`topics.jsonl` — первый запуск по заголовкам тезисов).

In [ ]:
import json, os, sys
from pathlib import Path

import yaml

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))  # если пакет не установлен через pip install -e .
from presentation_rl.tools.corpus import CorpusStore, TopicPlan, collect
from presentation_rl.tools.wiki import WikiClient

CONTACT = os.environ.get("WIKI_CONTACT", "")  # или впишите сюда почту / ссылку на проект
OUT = ROOT / "data" / "sources" / "wiki_pilot"

In [ ]:
# Темы и запросы: тема + ключевые слова тезисов (terms) всех трёх составителей без повторов.
topics = yaml.safe_load((ROOT / "data/tasks/pilot_topics.yaml").read_text(encoding="utf-8"))["topics"]
drafts = ["claude/pilot3.json", "codex/pilot3.json", "gemini/pilot3_notool.json"]
terms = {t["id"]: [] for t in topics}
for name in drafts:
    data = json.loads((ROOT / "data/tasks/drafts" / name).read_text(encoding="utf-8"))
    for t in data["topics"]:
        for point in t["key_points"]:
            for term in point["terms"]:
                if term.lower() not in (x.lower() for x in terms[t["id"]]):
                    terms[t["id"]].append(term)

plans = [TopicPlan(t["id"], t["topic"], terms[t["id"]]) for t in topics]
for p in plans:
    print(f"{p.id}: {len(p.queries)} запросов, например: {', '.join(p.queries[:5])}")

In [ ]:
client = WikiClient(contact=CONTACT)
store = CorpusStore(OUT, topics_name="topics_v2.jsonl")
collect(client, store, plans)

In [ ]:
# Что получилось.
articles = store.unique_articles()
by_lang = {lang: [a for a in articles if a.lang == lang] for lang in ("ru", "en")}
for lang, items in by_lang.items():
    sections = sum(len(a.sections) for a in items)
    chars = sum(len(s.text) for a in items for s in a.sections)
    print(f"{lang}: статей {len(items)}, разделов {sections}, символов {chars:,}")
print(f"размер articles.jsonl: {store.articles_path.stat().st_size / 1e6:.1f} МБ")
print()
for record in store.topics.values():
    print(f"{record['id']}: лучшие по теме {record['top_ru']}, ru {len(record['articles_ru'])}, en {len(record['articles_en'])}")

In [ ]:
# Оглавления лучших статей по теме на обоих языках.
for record in store.topics.values():
    for title in record["top_ru"][:2]:
        article = store.articles.get(("ru", title))
        if article:
            heads = [("  " * (s.level - 1)) + (s.heading or "(введение)") + f" [{len(s.text)}]" for s in article.sections]
            print(f"{record['id']} / ru: {article.title}")
            print("\n".join(heads[:20]))
            print()